# Notebook 13 — Neural Tabular Baseline (MLP) Comparison
## FYP: Adaptive Explainable Ensemble for Pre-Launch Steam Game Reception Prediction
### Heshan Ratnaweera | IIT Sri Lanka | W2082289 | 2026

**Purpose:** Benchmark a Multi-Layer Perceptron (MLP) neural network against the CatBoost/Random
Forest/XGBoost/Logistic Regression comparison already established in NB03, on the same T4
feature set. This answers Reviewer 1's comment: *"Additional comparisons with alternative model
families, such as neural tabular models... could strengthen the argument that the observed
results are due to the proposed routing strategy rather than model selection."*

**Expected outcome, stated up front:** the paper already cites Grinsztajn et al. [8] and Uddin &
Lu [9], both of which find tree-based models consistently outperform neural networks on tabular
data of this size and structure. This experiment is expected to reproduce that finding, which
would be a positive result for the paper (independent empirical confirmation of the literature
you already cite), not a null result. If the MLP happens to outperform CatBoost, that is also
useful to know before resubmission, better to find out now than have a reviewer point it out.

**This is a READ-ONLY analysis notebook with respect to existing project artefacts.** It trains a
new model (the MLP) but does not modify `games_features_t4.csv`, does not overwrite
`model_a/b/c/d/e.pkl`, and does not touch any file from NB01–NB11.

**Inputs (read-only):**
- `data/processed/games_features_t4.csv` (built in NB02)

**Outputs:** printed comparison table only, matching NB03's format so the row can be dropped
directly into the existing baseline table. Nothing is saved to disk unless you uncomment the
save cell at the end.

**A note on class imbalance:** unlike LogisticRegression, RandomForestClassifier, XGBoost, and
CatBoost, scikit-learn's `MLPClassifier` does not support `class_weight` or `sample_weight`
natively. To keep the imbalance handling broadly comparable to the rest of the project, this
notebook applies SMOTE (the same library already used in NB03 §6 and NB08 Experiment 2) inside
the cross-validation loop only, so the minority class is oversampled without ever leaking
information from validation folds into training folds.

---
## Contents
1. Setup and Imports
2. Load Processed Features and Recreate the Train/Test Split
3. Baseline MLP — 5-Fold CV (matching NB03's protocol exactly)
4. Hyperparameter Search — Tuned MLP
5. Final Test-Set Evaluation
6. Consolidated Comparison Table (drop-in replacement for the paper's baseline table)
7. Summary


## 1. Setup and Imports

In [1]:
import sys
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate, RandomizedSearchCV
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import make_scorer, f1_score, roc_auc_score, accuracy_score

try:
    from imblearn.pipeline import Pipeline as ImbPipeline
    from imblearn.over_sampling import SMOTE
    SMOTE_AVAILABLE = True
except ImportError:
    print('imbalanced-learn not available — install with: pip install imbalanced-learn --break-system-packages')
    SMOTE_AVAILABLE = False

warnings.filterwarnings('ignore')

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.config import TARGET_COL, RANDOM_STATE, FEATURES_T4_CSV, TIER_FEATURES

CV_FOLDS = 5   # matches NB03

f1_minority_scorer = make_scorer(f1_score, pos_label=0)
f1_majority_scorer = make_scorer(f1_score, pos_label=1)

print(f'Project root : {PROJECT_ROOT}')
print(f'Random state : {RANDOM_STATE}')
print(f'SMOTE available: {SMOTE_AVAILABLE}')


Project root : C:\Users\3214h\Documents\fyp-steam-reception
Random state : 42
SMOTE available: True


## 2. Load Processed Features and Recreate the Train/Test Split

Identical loading and split logic to NB03/NB04 — same file, same `test_size=0.2`, same
`random_state`. This guarantees the MLP is compared on exactly the same train/test partition as
every other model in the baseline table.


In [2]:
assert FEATURES_T4_CSV.exists(), f'Not found: {FEATURES_T4_CSV} — check the path matches your project layout.'

df = pd.read_csv(FEATURES_T4_CSV)
print(f'Loaded: {df.shape[0]:,} rows x {df.shape[1]} columns')
print(f'Using target column: {TARGET_COL}')

# Uses config.py's TIER_FEATURES[4] directly — verified via verify_tier_order.py
# to match the exact order model_d.pkl/model_e.pkl were trained on.
feature_cols = TIER_FEATURES[4]
missing = [c for c in feature_cols if c not in df.columns]
assert not missing, f'Columns missing from CSV: {missing}'
extra_cols = [c for c in df.columns if c not in feature_cols + [TARGET_COL]]
if extra_cols:
    print(f'Note: {len(extra_cols)} non-feature columns present in the CSV and ignored: {extra_cols}')

X = df[feature_cols].values
y = df[TARGET_COL].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

print(f'Train: {X_train.shape[0]:,} games   Test: {X_test.shape[0]:,} games')
print(f'Features: {X_train.shape[1]}')


Loaded: 20,383 rows x 56 columns
Using target column: reception_label
Note: 2 non-feature columns present in the CSV and ignored: ['dlc_count', 'short_description']
Train: 16,306 games   Test: 4,077 games
Features: 53


## 3. Baseline MLP — 5-Fold CV

Default architecture: two hidden layers (64, 32 units), ReLU activation, Adam solver, L2
regularisation (`alpha`) and early stopping in place of dropout (scikit-learn's `MLPClassifier`
does not implement dropout — if you want true dropout, a Keras/PyTorch model would be needed
instead, at the cost of a new dependency your `fenv` environment may not already have).

`StandardScaler` is required — unlike tree-based models, MLPs are sensitive to feature scale, the
same reason NB03 scaled inputs for Logistic Regression.


In [3]:
def build_mlp_pipeline(hidden_layer_sizes=(64, 32), alpha=0.0001, learning_rate_init=0.001,
                       activation='relu'):
    steps = [('scaler', StandardScaler())]
    if SMOTE_AVAILABLE:
        steps.append(('smote', SMOTE(random_state=RANDOM_STATE)))
    steps.append(('clf', MLPClassifier(
        hidden_layer_sizes=hidden_layer_sizes,
        activation=activation,
        solver='adam',
        alpha=alpha,
        learning_rate_init=learning_rate_init,
        max_iter=500,
        early_stopping=True,
        n_iter_no_change=15,
        random_state=RANDOM_STATE,
    )))
    PipelineClass = ImbPipeline if SMOTE_AVAILABLE else __import__('sklearn.pipeline', fromlist=['Pipeline']).Pipeline
    return PipelineClass(steps)


print('Running 5-fold stratified CV on training set '
      f'({X_train.shape[0]:,} games)...')
print('Baseline MLP: hidden_layer_sizes=(64, 32)')
print()

cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)

start = time.time()
mlp_baseline = build_mlp_pipeline()
scores_mlp = cross_validate(
    mlp_baseline, X_train, y_train, cv=cv,
    scoring={'f1_macro': 'f1_macro', 'f1_minority': f1_minority_scorer,
             'f1_majority': f1_majority_scorer, 'roc_auc': 'roc_auc', 'accuracy': 'accuracy'},
    n_jobs=1
)
elapsed_baseline = time.time() - start

print(f'Training MLP (baseline)... done in {elapsed_baseline:.1f}s  |  '
      f'Macro F1={scores_mlp["test_f1_macro"].mean():.4f} ± {scores_mlp["test_f1_macro"].std():.4f}  |  '
      f'Minority F1={scores_mlp["test_f1_minority"].mean():.4f}  |  '
      f'AUC={scores_mlp["test_roc_auc"].mean():.4f}  |  '
      f'Accuracy={scores_mlp["test_accuracy"].mean():.4f}')


Running 5-fold stratified CV on training set (16,306 games)...
Baseline MLP: hidden_layer_sizes=(64, 32)

Training MLP (baseline)... done in 36.9s  |  Macro F1=0.6165 ± 0.0143  |  Minority F1=0.4674  |  AUC=0.6726  |  Accuracy=0.6746


## 4. Hyperparameter Search — Tuned MLP

Matching NB03's approach for Random Forest and CatBoost: a `RandomizedSearchCV` over the key
architecture and optimisation hyperparameters, 5-fold CV, scored on Macro F1. Neural network
tuning is slower than tree-based tuning per trial — `n_iter=15` is set conservatively; reduce it
if this takes too long on your hardware, or increase it if you have time for a more thorough
search.


In [4]:
from scipy.stats import uniform, loguniform

print('=== RANDOMIZED HYPERPARAMETER SEARCH (MLP) ===')
print('Searching over architecture and optimisation hyperparameters using macro F1 as target...')
print('(n_iter=15, 5-fold CV — this may take several minutes)')
print()

mlp_param_dist = {
    'clf__hidden_layer_sizes': [(32,), (64,), (64, 32), (128, 64), (100, 50, 25)],
    'clf__alpha':              loguniform(1e-5, 1e-1),
    'clf__learning_rate_init': loguniform(1e-4, 1e-2),
    'clf__activation':         ['relu', 'tanh'],
}

base_pipeline = build_mlp_pipeline()

mlp_search = RandomizedSearchCV(
    base_pipeline,
    mlp_param_dist,
    n_iter=15,
    scoring='f1_macro',
    cv=cv,
    random_state=RANDOM_STATE,
    n_jobs=1,
    verbose=0,
)
mlp_search.fit(X_train, y_train)

mlp_tuned_macro = mlp_search.best_score_
print(f'MLP tuned:')
print(f'  Best params : {mlp_search.best_params_}')
print(f'  CV Macro F1 : {mlp_tuned_macro:.4f}  (baseline: {scores_mlp["test_f1_macro"].mean():.4f})')


=== RANDOMIZED HYPERPARAMETER SEARCH (MLP) ===
Searching over architecture and optimisation hyperparameters using macro F1 as target...
(n_iter=15, 5-fold CV — this may take several minutes)

MLP tuned:
  Best params : {'clf__activation': 'relu', 'clf__alpha': np.float64(0.0012563152773938662), 'clf__hidden_layer_sizes': (64,), 'clf__learning_rate_init': np.float64(0.00012397420340784152)}
  CV Macro F1 : 0.6382  (baseline: 0.6165)


## 5. Final Test-Set Evaluation

Evaluate the tuned MLP on the held-out test set, directly comparable to Model D's test Macro F1 = 0.6690 and CatBoost-tuned's CV Macro F1 = 0.6695 from NB03.

In [5]:
mlp_final = mlp_search.best_estimator_
mlp_final.fit(X_train, y_train)

y_pred_mlp = mlp_final.predict(X_test)
y_prob_mlp = mlp_final.predict_proba(X_test)[:, 1]

test_macro_mlp = f1_score(y_test, y_pred_mlp, average='macro')
test_minor_mlp = f1_score(y_test, y_pred_mlp, pos_label=0)
test_auc_mlp   = roc_auc_score(y_test, y_prob_mlp)
test_acc_mlp   = accuracy_score(y_test, y_pred_mlp)

print('=== MLP (TUNED) — TEST SET RESULTS ===')
print(f'  Macro F1    : {test_macro_mlp:.4f}')
print(f'  Minority F1 : {test_minor_mlp:.4f}')
print(f'  AUC-ROC     : {test_auc_mlp:.4f}')
print(f'  Accuracy    : {test_acc_mlp:.4f}')
print()
print(f'  For reference — Model D (CatBoost, T4, from NB04): Macro F1 = 0.6690')
print(f'  For reference — Uncertainty router (D+E, from NB04): Macro F1 = 0.6796')


=== MLP (TUNED) — TEST SET RESULTS ===
  Macro F1    : 0.6530
  Minority F1 : 0.5323
  AUC-ROC     : 0.7266
  Accuracy    : 0.6949

  For reference — Model D (CatBoost, T4, from NB04): Macro F1 = 0.6690
  For reference — Uncertainty router (D+E, from NB04): Macro F1 = 0.6796


## 6. Consolidated Comparison Table

This reproduces NB03's baseline table format with the MLP rows appended, ready to drop directly
into the paper. The LR/RF/XGBoost/CatBoost numbers below are hardcoded from NB03's verified
output (not re-run here) purely for side-by-side display — your MLP numbers above are freshly
computed in this notebook.


In [6]:
print('=== FULL BASELINE COMPARISON (5-fold CV, training set) ===')
print(f'{"Model":<24} {"CV Macro F1":>14} {"CV Minority F1":>16} {"CV AUC":>10} {"Train time":>12}')
print('-' * 78)
print(f'{"Logistic Regression":<24} {0.6311:>14.4f} {0.5218:>16.4f} {0.7158:>10.4f} {"0.4s":>12}')
print(f'{"Random Forest (default)":<24} {0.6677:>14.4f} {0.5243:>16.4f} {0.7444:>10.4f} {"9.0s":>12}')
print(f'{"XGBoost (default)":<24} {0.6591:>14.4f} {0.5390:>16.4f} {0.7379:>10.4f} {"3.1s":>12}')
print(f'{"CatBoost (default)":<24} {0.6580:>14.4f} {0.5438:>16.4f} {0.7443:>10.4f} {"12.8s":>12}')
print(f'{"Random Forest (tuned)":<24} {0.6644:>14.4f} {0.5336:>16.4f} {"-":>10} {"-":>12}')
print(f'{"CatBoost (tuned)":<24} {0.6695:>14.4f} {0.5313:>16.4f} {"-":>10} {"-":>12}')
print(f'{"MLP (baseline, this NB)":<24} {scores_mlp["test_f1_macro"].mean():>14.4f} '
      f'{scores_mlp["test_f1_minority"].mean():>16.4f} {scores_mlp["test_roc_auc"].mean():>10.4f} '
      f'{elapsed_baseline:>11.1f}s')
print(f'{"MLP (tuned, this NB)":<24} {mlp_tuned_macro:>14.4f} {"see test set":>16} {"-":>10} {"-":>12}')
print()
print(f'{"MLP (tuned) TEST SET":<24} {test_macro_mlp:>14.4f} {test_minor_mlp:>16.4f} {test_auc_mlp:>10.4f} {"-":>12}')


=== FULL BASELINE COMPARISON (5-fold CV, training set) ===
Model                       CV Macro F1   CV Minority F1     CV AUC   Train time
------------------------------------------------------------------------------
Logistic Regression              0.6311           0.5218     0.7158         0.4s
Random Forest (default)          0.6677           0.5243     0.7444         9.0s
XGBoost (default)                0.6591           0.5390     0.7379         3.1s
CatBoost (default)               0.6580           0.5438     0.7443        12.8s
Random Forest (tuned)            0.6644           0.5336          -            -
CatBoost (tuned)                 0.6695           0.5313          -            -
MLP (baseline, this NB)          0.6165           0.4674     0.6726        36.9s
MLP (tuned, this NB)             0.6382     see test set          -            -

MLP (tuned) TEST SET             0.6530           0.5323     0.7266            -


## 7. Summary

Compare `MLP (tuned) TEST SET` above against Model D's test Macro F1 = 0.6690. If the MLP scores
lower (expected, per Grinsztajn et al. [8] and Uddin & Lu [9]), this becomes a clean one-sentence
addition to the Discussion: *"A tuned MLP baseline on the same T4 features reached Macro F1 =
[value], below CatBoost's 0.6690, consistent with the broader finding that tree-based models
outperform neural networks on tabular data of this scale [8][9]."*

If the MLP scores higher, do not discard this, it is a genuinely useful finding to know before
resubmission and would need a different framing in the paper. Either outcome is useful; that is
why this experiment was worth running.

Nothing in NB01–NB11 was modified by running this notebook.
